# ASKAP Holography HDF5 Analysis
## Per-antenna beam characterisation — SB82194

This notebook mirrors the FITS-based `holopy_demo.ipynb` but operates directly on the
**native HDF5 pipeline products** where the full antenna dimension is preserved.

### Key difference from the FITS analysis

| Aspect | FITS (`holopy_demo`) | HDF5 (this notebook) |
|---|---|---|
| Pixel grid | 101 × 101 (WCS, arcmin) | 15 × 15 (holography grid, radians) |
| Antenna axis | Collapsed (not in file) | Full: 36 antennas retained |
| Polarisations | Stokes IQUV (real, float) | Raw XX/XY/YX/YY (complex) **and** Stokes IQUV (float) |
| Flags | NaN masking only | Explicit boolean flag dataset |
| Data type | float32 | complex64 (raw) / float64 (Stokes) |

### HDF5 files used
| File | Shape | Content |
|---|---|---|
| `*.xxyy…hdf5` | `(1, 36 ant, 36 beam, 4 pol, 288 freq, 15, 15)` | Raw complex XX/XY/YX/YY cross-correlations |
| `*.stokes…hdf5` | `(1, 36 ant, 36 beam, 4 pol, 288 freq, 15, 15)` | Normalised Stokes IQUV (float64) |

Axis order in memory (after loading): `[time, antenna, beam, pol, freq, y, x]`

### Analysis sections
1. Imports & file paths
2. Load & inspect HDF5 structure
3. Flag statistics
4. Global statistics (alive check)
5. Cross-channel consistency per antenna
6. Per-antenna beam maps — all beams, single Stokes
7. Antenna–beam amplitude matrix (heatmap)
8. FWHM estimation across antennas
9. Centroid offsets across antennas
10. Stokes IQUV panels per antenna
11. Frequency spectra per antenna
12. Channel flagging per antenna
13. Antenna-to-antenna difference maps
14. Cross-antenna outlier detection
15. Raw XX/YY — amplitude and phase inspection

## 1. Imports & File Paths

In [2]:
import sys, os
import numpy as np
import h5py
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import center_of_mass
from scipy.optimize import curve_fit

DATA_DIR = "/Users/raj030/DATA/askap-beam-analysis"

XXYY_FILE   = os.path.join(DATA_DIR, "akpb.xxyy.closepack36.54.943MHz.SB82194.hdf5")
STOKES_FILE = os.path.join(DATA_DIR, "akpb.iquv.closepack36.54.943MHz.SB82194.stokes.hdf5")

print("Imports OK")
for f in [XXYY_FILE, STOKES_FILE]:
    size_gb = os.path.getsize(f) / 1e9
    print(f"  {os.path.basename(f)}  ({size_gb:.2f} GB)")

ModuleNotFoundError: No module named 'h5py'

## 2. Load & Inspect HDF5 Structure

Both files follow the same internal schema:
- **`data`** dataset — the beam measurements with shape `(time, antenna, beam, pol, freq, y, x)`
- **`flags`** dataset — boolean mask, same shape (or collapsed spatial axes)
- Rich **attributes** on the `data` dataset: frequency array, x/y axes (radians), antenna/beam lists, polarisation labels, history, epoch, SB IDs

We load the Stokes HDF5 into memory (2.5 GB float64) and keep a reference to the raw XXYY file
for complex visibility inspection later.

In [ ]:
# ── Load Stokes HDF5 ──────────────────────────────────────────────
with h5py.File(STOKES_FILE, 'r') as f:
    stokes_data  = f['data'][0]           # drop time axis → (ant, beam, pol, freq, y, x)
    stokes_flags = f['flags'][0]          # (ant, beam, pol, freq) or (ant, beam, pol, freq, y, x)
    freqs_mhz    = f['data'].attrs['frequencies']          # (288,) MHz
    antennas     = f['data'].attrs['antennas']             # (36,) 1-based
    beams        = f['data'].attrs['beams']                # (36,) 1-based
    pols_stokes  = [p.decode() if isinstance(p, bytes) else p
                    for p in f['data'].attrs['polarizations']]  # ['I','Q','U','V']
    x_axis_rad   = f['data'].attrs['xAxis']                # (15,) radians
    y_axis_rad   = f['data'].attrs['yAxis']                # (15,) radians

N_ANT, N_BEAM, N_POL, N_FREQ, NY, NX = stokes_data.shape
POL_IDX = {p: i for i, p in enumerate(pols_stokes)}

x_axis_deg = np.degrees(x_axis_rad)
y_axis_deg = np.degrees(y_axis_rad)
extent_deg = [x_axis_deg[0], x_axis_deg[-1], y_axis_deg[0], y_axis_deg[-1]]

print(f"Stokes data shape  : {stokes_data.shape}")
print(f"  axes             : (antenna, beam, pol, freq, y, x)")
print(f"  antennas         : {N_ANT}  {antennas}")
print(f"  beams            : {N_BEAM}")
print(f"  polarisations    : {pols_stokes}")
print(f"  freq channels    : {N_FREQ}  ({freqs_mhz[0]:.2f} – {freqs_mhz[-1]:.2f} MHz)")
print(f"  spatial grid     : {NY} x {NX}  ({np.degrees(x_axis_rad[-1]-x_axis_rad[0]):.2f}° FOV)")
print(f"Flags shape        : {stokes_flags.shape}")

## 3. Flag Statistics

The HDF5 files carry an explicit `flags` boolean dataset.  
We summarise flagging per antenna and per beam to identify antennas or beams with
disproportionate data loss.

In [ ]:
# flags shape: (ant, beam, pol, freq)  — no spatial axes
# Collapse over beam, pol, freq → per-antenna flagging fraction
flag_per_ant  = stokes_flags.mean(axis=(1, 2, 3))   # (N_ANT,)
flag_per_beam = stokes_flags.mean(axis=(0, 2, 3))   # (N_BEAM,)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(antennas, flag_per_ant * 100, color='steelblue', edgecolor='k', linewidth=0.4)
axes[0].set_xlabel('Antenna number')
axes[0].set_ylabel('Flagged fraction (%)')
axes[0].set_title('Flag fraction per antenna (all beams, pols, channels)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(beams, flag_per_beam * 100, color='tomato', edgecolor='k', linewidth=0.4)
axes[1].set_xlabel('Beam number')
axes[1].set_ylabel('Flagged fraction (%)')
axes[1].set_title('Flag fraction per beam (all antennas, pols, channels)')
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Overall flag fraction : {stokes_flags.mean()*100:.2f}%")
print(f"Most flagged antenna  : {antennas[np.argmax(flag_per_ant)]}  ({flag_per_ant.max()*100:.1f}%)")
print(f"Most flagged beam     : {beams[np.argmax(flag_per_beam)]}  ({flag_per_beam.max()*100:.1f}%)")

## 4. Global Statistics (Alive Check)

Analogous to Section 7 of the FITS notebook: pool all finite Stokes-I values across
all antennas, beams, channels, and spatial pixels and compute scalar statistics.

> This is deliberately coarse — inter-antenna and inter-channel structure is addressed
> in later sections.

In [ ]:
stokes_I = stokes_data[:, :, POL_IDX['I'], :, :, :]   # (ant, beam, freq, y, x)

flat = stokes_I[np.isfinite(stokes_I)].ravel()
stats = {
    'min':    float(flat.min()),
    'max':    float(flat.max()),
    'mean':   float(flat.mean()),
    'median': float(np.median(flat)),
    'std':    float(flat.std()),
    'p5':     float(np.percentile(flat,  5)),
    'p95':    float(np.percentile(flat, 95)),
    'p99':    float(np.percentile(flat, 99)),
}
print('Global Stokes-I statistics (all antennas, beams, channels, pixels):')
for k, v in stats.items():
    print(f'  {k:<8}: {v:.6g}')

## 5. Cross-Channel Consistency per Antenna

For each antenna and each frequency channel we compute the **peak Stokes-I amplitude**
averaged over all 36 beams.  This gives a `(36 ant, 288 freq)` surface that reveals:

- Antennas with unusual bandpass shapes
- Channels that are bad for specific antennas (direction-dependent RFI or calibration failures)

The heatmap shows this surface; the line plot below shows the per-antenna mean spectrum.

In [ ]:
# peak amplitude per (antenna, channel): average over beams then max over spatial
# shape: (N_ANT, N_FREQ)
peak_ant_chan = np.nanmean(
    np.nanmax(stokes_I, axis=(-2, -1)),   # max over y,x → (ant, beam, freq)
    axis=1                                 # mean over beams → (ant, freq)
)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Heatmap: antenna vs frequency
im = axes[0].imshow(
    peak_ant_chan, aspect='auto', origin='lower',
    extent=[freqs_mhz[0], freqs_mhz[-1], 0.5, N_ANT + 0.5],
    cmap='inferno'
)
axes[0].set_ylabel('Antenna index')
axes[0].set_title('Stokes-I peak amplitude — antenna × frequency heatmap')
plt.colorbar(im, ax=axes[0], label='Peak amplitude')

# Overlay mean spectrum (collapsed over antennas)
for i in range(N_ANT):
    axes[1].plot(freqs_mhz, peak_ant_chan[i], lw=0.6, alpha=0.5, label=f'Ant {antennas[i]}')
axes[1].plot(freqs_mhz, peak_ant_chan.mean(axis=0), lw=1.8, color='k', label='Mean')
axes[1].set_xlabel('Frequency (MHz)')
axes[1].set_ylabel('Peak amplitude')
axes[1].set_title('Per-antenna Stokes-I spectra (beam-averaged peak)')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Per-Antenna Beam Maps

For a chosen antenna we show the frequency-averaged Stokes-I beam map for **all 36 beams**
in a 6×6 mosaic — identical layout to Section 9 of the FITS notebook but now we can
cycle through `ANT_IDX` to compare different antennas.

Set `ANT_IDX` (0-based) to select the antenna.

In [ ]:
ANT_IDX = 0   # ← change to inspect a different antenna (0-based)

# frequency-average over channels → (beam, y, x)
avg_maps = np.nanmean(stokes_data[ANT_IDX, :, POL_IDX['I'], :, :, :], axis=1)  # (36, 15, 15)

fig, axes = plt.subplots(6, 6, figsize=(14, 14))
axes = axes.ravel()

for b in range(N_BEAM):
    img = avg_maps[b]
    peak = np.nanmax(img)
    img_norm = img / peak if peak > 0 else img
    axes[b].imshow(
        img_norm, origin='lower', cmap='inferno',
        extent=[x_axis_deg[0], x_axis_deg[-1], y_axis_deg[0], y_axis_deg[-1]],
        vmin=0, vmax=1
    )
    axes[b].set_title(f'B{beams[b]}', fontsize=7)
    axes[b].axis('off')

fig.suptitle(
    f'Ant {antennas[ANT_IDX]} — Stokes I, all 36 beams (freq-averaged, self-normalised)',
    fontsize=11, y=1.01
)
fig.tight_layout()
plt.show()

## 7. Antenna–Beam Amplitude Matrix

A single heatmap of shape `(36 ant, 36 beam)` where each cell shows the
**peak, frequency-averaged, Stokes-I amplitude**.  This is the primary diagnostic
for:
- Antennas with systematically lower/higher gain
- Beams that are anomalously weak or strong for *specific* antennas
- Structured patterns (e.g. rows of low gain) that point to array-level failures

In [ ]:
# peak over spatial, mean over frequency → (ant, beam)
peak_mat = np.nanmax(
    np.nanmean(stokes_data[:, :, POL_IDX['I'], :, :, :], axis=2),   # mean freq → (ant,beam,y,x)
    axis=(-2, -1)                                                      # max spatial → (ant,beam)
)

fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(peak_mat, aspect='auto', cmap='inferno', origin='upper')
plt.colorbar(im, ax=ax, label='Peak Stokes-I amplitude')
ax.set_xticks(np.arange(N_BEAM))
ax.set_xticklabels(beams, fontsize=6, rotation=45)
ax.set_yticks(np.arange(N_ANT))
ax.set_yticklabels(antennas, fontsize=7)
ax.set_xlabel('Beam number')
ax.set_ylabel('Antenna number')
ax.set_title('Antenna × Beam peak Stokes-I amplitude (freq-averaged)')

# annotate outliers (> 3σ from mean)
mu, sig = peak_mat.mean(), peak_mat.std()
outliers = np.argwhere(np.abs(peak_mat - mu) > 3 * sig)
for (i, j) in outliers:
    ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                               fill=False, edgecolor='cyan', lw=1.5))

fig.tight_layout()
plt.show()

print(f'Mean peak amplitude : {mu:.4f}  ±  {sig:.4f}')
print(f'Outliers (>3σ)      : {len(outliers)}  cells circled in cyan')

## 8. FWHM Estimation Across Antennas

We estimate the FWHM for every `(antenna, beam)` pair by finding the half-power
crossing along the x and y axes of the frequency-averaged Stokes-I map.  The results
are compared as a box-plot per beam (colour-coded by antenna) and as a
`(antenna, beam)` heatmap analogous to Section 16 of the FITS notebook.

In [ ]:
def fwhm_1d(profile, axis_deg):
    """Half-power FWHM in degrees for a 1-D profile."""
    peak = np.nanmax(profile)
    if peak <= 0:
        return np.nan
    half = peak / 2.0
    above = profile >= half
    if not above.any():
        return np.nan
    idxs = np.where(above)[0]
    return float(axis_deg[idxs[-1]] - axis_deg[idxs[0]])

# freq-averaged Stokes-I: (ant, beam, y, x)
avg_I = np.nanmean(stokes_data[:, :, POL_IDX['I'], :, :, :], axis=2)

fwhm_x = np.full((N_ANT, N_BEAM), np.nan)
fwhm_y = np.full((N_ANT, N_BEAM), np.nan)

for a in range(N_ANT):
    for b in range(N_BEAM):
        img = avg_I[a, b]
        cy, cx = img.shape[0]//2, img.shape[1]//2
        fwhm_x[a, b] = fwhm_1d(img[cy, :], x_axis_deg)
        fwhm_y[a, b] = fwhm_1d(img[:, cx], y_axis_deg)

fwhm_mean = (fwhm_x + fwhm_y) / 2.0

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

im0 = axes[0].imshow(fwhm_mean * 60, aspect='auto', cmap='viridis', origin='upper')
plt.colorbar(im0, ax=axes[0], label='Mean FWHM (arcmin)')
axes[0].set_xticks(np.arange(N_BEAM))
axes[0].set_xticklabels(beams, fontsize=6, rotation=45)
axes[0].set_yticks(np.arange(N_ANT))
axes[0].set_yticklabels(antennas, fontsize=7)
axes[0].set_xlabel('Beam'); axes[0].set_ylabel('Antenna')
axes[0].set_title('Mean FWHM (arcmin) — antenna × beam')

# Box-plot: distribution of FWHM across antennas, per beam
axes[1].boxplot(
    [fwhm_mean[:, b] * 60 for b in range(N_BEAM)],
    positions=np.arange(N_BEAM),
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6)
)
axes[1].set_xticks(np.arange(N_BEAM))
axes[1].set_xticklabels(beams, fontsize=6, rotation=45)
axes[1].set_xlabel('Beam')
axes[1].set_ylabel('FWHM (arcmin)')
axes[1].set_title('FWHM spread across antennas per beam')
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

print(f'Overall median FWHM : {np.nanmedian(fwhm_mean)*60:.1f} arcmin')
print(f'Std across all cells: {np.nanstd(fwhm_mean)*60:.2f} arcmin')

## 9. Centroid Offsets Across Antennas

The intensity-weighted centroid of each `(antenna, beam)` map is computed in degrees.
Offsets from zero reveal pointing or beam-steering variations between antennas.

A quiver plot shows centroid displacement vectors; the colour encodes displacement
magnitude.

In [ ]:
cx_deg = np.full((N_ANT, N_BEAM), np.nan)
cy_deg = np.full((N_ANT, N_BEAM), np.nan)

for a in range(N_ANT):
    for b in range(N_BEAM):
        img = avg_I[a, b]
        img_pos = np.where(img > 0, img, 0.0)
        total = img_pos.sum()
        if total > 0:
            yc, xc = center_of_mass(img_pos)
            # interpolate to axis values
            cx_deg[a, b] = np.interp(xc, np.arange(NX), x_axis_deg)
            cy_deg[a, b] = np.interp(yc, np.arange(NY), y_axis_deg)

displace = np.sqrt(cx_deg**2 + cy_deg**2)  # (ant, beam) angular displacement in degrees

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(displace * 60, aspect='auto', cmap='plasma', origin='upper')
plt.colorbar(im1, ax=axes[0], label='Centroid displacement (arcmin)')
axes[0].set_xticks(np.arange(N_BEAM))
axes[0].set_xticklabels(beams, fontsize=6, rotation=45)
axes[0].set_yticks(np.arange(N_ANT))
axes[0].set_yticklabels(antennas, fontsize=7)
axes[0].set_xlabel('Beam'); axes[0].set_ylabel('Antenna')
axes[0].set_title('Centroid displacement from phase centre (arcmin)')

# per-antenna mean centroid offset (averaged over beams)
mean_cx = np.nanmean(cx_deg, axis=1) * 60   # arcmin
mean_cy = np.nanmean(cy_deg, axis=1) * 60
sc = axes[1].scatter(mean_cx, mean_cy,
                     c=antennas, cmap='tab20', s=60, zorder=3)
plt.colorbar(sc, ax=axes[1], label='Antenna number')
for i, ant in enumerate(antennas):
    axes[1].annotate(str(ant), (mean_cx[i], mean_cy[i]), fontsize=6,
                     textcoords='offset points', xytext=(3, 3))
axes[1].axhline(0, color='k', lw=0.5, ls='--')
axes[1].axvline(0, color='k', lw=0.5, ls='--')
axes[1].set_xlabel('Mean Δx (arcmin)')
axes[1].set_ylabel('Mean Δy (arcmin)')
axes[1].set_title('Per-antenna mean centroid offset (averaged over all beams)')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 10. Stokes IQUV Panels per Antenna

For a chosen `(antenna, beam)` pair, display the four Stokes maps side by side —
identical layout to Section 10 of the FITS notebook.  Change `ANT_IDX` and
`BEAM_IDX` to navigate the full dataset.

In [ ]:
ANT_IDX  = 0   # ← 0-based; antenna number = antennas[ANT_IDX]
BEAM_IDX = 0   # ← 0-based; beam number   = beams[BEAM_IDX]

# freq-average: (pol, y, x)
stokes_maps = np.nanmean(stokes_data[ANT_IDX, BEAM_IDX, :, :, :, :], axis=1)   # (4, 15, 15)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cmaps = ['inferno', 'RdBu_r', 'RdBu_r', 'RdBu_r']

for i, (stk, cmap) in enumerate(zip(pols_stokes, cmaps)):
    img = stokes_maps[i]
    vmax = np.nanpercentile(np.abs(img), 99)
    vmin = 0 if stk == 'I' else -vmax
    im = axes[i].imshow(
        img, origin='lower', cmap=cmap,
        extent=extent_deg, vmin=vmin, vmax=vmax
    )
    plt.colorbar(im, ax=axes[i], shrink=0.85)
    axes[i].set_title(f'Stokes {stk}')
    axes[i].set_xlabel('Δx (deg)')
    if i == 0:
        axes[i].set_ylabel('Δy (deg)')

fig.suptitle(
    f'Antenna {antennas[ANT_IDX]}, Beam {beams[BEAM_IDX]} — Stokes IQUV (freq-averaged)',
    y=1.01
)
fig.tight_layout()
plt.show()

## 11. Frequency Spectra per Antenna

For a selected beam, the peak Stokes-I amplitude **per frequency channel** is plotted
for every antenna on the same axes.  Diverging antennas or channels stand out clearly.

The coefficient of variation (CV) across antennas per channel (bottom panel) flags
channels where antennas disagree — analogous to Section 7b of the FITS notebook but
now the scatter is *across antennas* rather than across formed beams.

In [ ]:
BEAM_IDX = 0   # ← beam to inspect

# peak over spatial, per (ant, freq): shape (N_ANT, N_FREQ)
spec = np.nanmax(
    stokes_data[:, BEAM_IDX, POL_IDX['I'], :, :, :],
    axis=(-2, -1)
)  # (N_ANT, N_FREQ)

chan_mean = np.nanmean(spec, axis=0)
chan_std  = np.nanstd(spec,  axis=0)
chan_cv   = 100.0 * chan_std / np.where(chan_mean > 0, chan_mean, np.nan)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

cmap_lines = plt.get_cmap('tab20')
for i in range(N_ANT):
    axes[0].plot(freqs_mhz, spec[i], lw=0.7, alpha=0.6,
                 color=cmap_lines(i / N_ANT), label=f'Ant {antennas[i]}')
axes[0].plot(freqs_mhz, chan_mean, 'k-', lw=1.5, label='Mean')
axes[0].set_ylabel('Peak Stokes-I')
axes[0].set_title(f'Beam {beams[BEAM_IDX]} — per-antenna Stokes-I spectrum')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=5, ncol=6, loc='upper right')

axes[1].plot(freqs_mhz, chan_cv, lw=0.9, color='tomato')
axes[1].axhline(5, ls='--', color='grey', lw=0.8, label='5% threshold')
axes[1].set_ylabel('CV across antennas (%)')
axes[1].set_xlabel('Frequency (MHz)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f'Median CV : {np.nanmedian(chan_cv):.2f}%')
print(f'Max CV    : {np.nanmax(chan_cv):.2f}% at {freqs_mhz[np.nanargmax(chan_cv)]:.1f} MHz')

## 12. Channel Flagging per Antenna

Using the explicit `flags` dataset we show — for a chosen beam — which channels are
flagged for each antenna.  A 2-D image `(antenna, channel)` with flagged cells
highlighted gives an immediate overview of the RFI/calibration environment.

In [ ]:
BEAM_IDX = 0

# flags shape: (ant, beam, pol, freq)
# Any-pol flagging per (ant, freq)
flag_slice = stokes_flags[:, BEAM_IDX, :, :].any(axis=1)   # (N_ANT, N_FREQ)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(
    flag_slice.astype(float), aspect='auto', cmap='Reds',
    origin='upper',
    extent=[freqs_mhz[0], freqs_mhz[-1], N_ANT + 0.5, 0.5]
)
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Antenna index')
ax.set_yticks(np.arange(1, N_ANT + 1))
ax.set_yticklabels(antennas, fontsize=7)
ax.set_title(f'Flagged channels per antenna — Beam {beams[BEAM_IDX]} (red = flagged)')
fig.tight_layout()
plt.show()

flagged_chans = flag_slice.any(axis=0).sum()
print(f'Channels flagged for ≥1 antenna : {flagged_chans} / {N_FREQ}')

## 13. Antenna-to-Antenna Difference Maps

Select a **reference antenna** and display the fractional difference
`(ant_i − ref) / ref` for all other antennas for a given beam.  This
directly quantifies how much the beam patterns deviate from antenna to antenna.

In [ ]:
REF_ANT  = 0   # reference antenna index (0-based)
BEAM_IDX = 0

ref_map = avg_I[REF_ANT, BEAM_IDX]   # (y, x)

# show first 8 non-reference antennas for brevity
compare_ants = [a for a in range(N_ANT) if a != REF_ANT][:8]
ncols = 4
nrows = int(np.ceil(len(compare_ants) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
axes = axes.ravel()

for idx, a in enumerate(compare_ants):
    diff = (avg_I[a, BEAM_IDX] - ref_map) / np.where(ref_map > 0, ref_map, np.nan)
    vmax = np.nanpercentile(np.abs(diff), 98)
    im = axes[idx].imshow(
        diff * 100, origin='lower', cmap='RdBu_r',
        extent=extent_deg, vmin=-vmax*100, vmax=vmax*100
    )
    plt.colorbar(im, ax=axes[idx], label='% diff')
    axes[idx].set_title(f'Ant {antennas[a]} − Ant {antennas[REF_ANT]}')
    axes[idx].set_xlabel('Δx (deg)')

for idx in range(len(compare_ants), len(axes)):
    axes[idx].axis('off')

fig.suptitle(
    f'Beam {beams[BEAM_IDX]} — fractional difference vs Antenna {antennas[REF_ANT]} (Stokes I, freq-avg)',
    y=1.01
)
fig.tight_layout()
plt.show()

## 14. Cross-Antenna Outlier Detection

For each beam, the distribution of peak Stokes-I amplitudes across the 36 antennas
is summarised as a box-plot.  Antennas outside 2×IQR are flagged as outliers.

A ranked bar chart lists the antennas by their mean outlier count across all beams.

In [ ]:
# peak_mat shape: (N_ANT, N_BEAM)  — computed in Section 7

outlier_count = np.zeros(N_ANT, dtype=int)
for b in range(N_BEAM):
    col = peak_mat[:, b]
    q1, q3 = np.nanpercentile(col, 25), np.nanpercentile(col, 75)
    iqr = q3 - q1
    outlier_count += (col < q1 - 2*iqr) | (col > q3 + 2*iqr)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box-plot
bp = axes[0].boxplot(
    [peak_mat[:, b] for b in range(N_BEAM)],
    positions=np.arange(N_BEAM),
    widths=0.6,
    patch_artist=True,
    flierprops=dict(marker='o', markersize=4, color='tomato'),
    boxprops=dict(facecolor='steelblue', alpha=0.5)
)
axes[0].set_xticks(np.arange(N_BEAM))
axes[0].set_xticklabels(beams, fontsize=6, rotation=45)
axes[0].set_xlabel('Beam')
axes[0].set_ylabel('Peak Stokes-I')
axes[0].set_title('Distribution of peak Stokes-I across antennas per beam')
axes[0].grid(axis='y', alpha=0.3)

# Ranks
sort_idx = np.argsort(-outlier_count)
axes[1].bar(np.arange(N_ANT), outlier_count[sort_idx],
            color=['tomato' if outlier_count[i] > 0 else 'steelblue' for i in sort_idx])
axes[1].set_xticks(np.arange(N_ANT))
axes[1].set_xticklabels(antennas[sort_idx], fontsize=7, rotation=45)
axes[1].set_xlabel('Antenna (ranked)')
axes[1].set_ylabel('Outlier beam count')
axes[1].set_title('Antennas ranked by number of beams where they are outliers')
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

print('Top outlier antennas:')
for i in sort_idx[:5]:
    print(f'  Antenna {antennas[i]:3d}  —  {outlier_count[i]} beam(s) outlier')

## 15. Raw XX/YY — Amplitude and Phase Inspection

The XXYY HDF5 contains the raw **complex** cross-correlations before Stokes
conversion.  Here we inspect amplitude and phase of the XX and YY diagonal
terms for a chosen `(antenna, beam)` pair.

Key things to check:
- XX and YY amplitudes should be similar (|XX|≈|YY|); large differences suggest
  a polarisation gain imbalance on that antenna.
- Phase should be smooth and slowly varying with frequency; rapid phase jumps
  indicate decorrelation or calibration failures.

In [ ]:
ANT_IDX  = 0
BEAM_IDX = 0

with h5py.File(XXYY_FILE, 'r') as f:
    pols_xxyy = [p.decode() if isinstance(p, bytes) else p
                 for p in f['data'].attrs['polarizations']]  # ['XX','XY','YX','YY']
    xxyy_slice = f['data'][0, ANT_IDX, BEAM_IDX, :, :, :, :]  # (pol, freq, y, x)

XX = xxyy_slice[pols_xxyy.index('XX')]   # (freq, y, x)
YY = xxyy_slice[pols_xxyy.index('YY')]

# Centre pixel
cy, cx = NY//2, NX//2
xx_spec = XX[:, cy, cx]   # complex spectrum at centre pixel
yy_spec = YY[:, cy, cx]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Amplitude
axes[0, 0].plot(freqs_mhz, np.abs(xx_spec), lw=0.9, label='|XX|', color='steelblue')
axes[0, 0].plot(freqs_mhz, np.abs(yy_spec), lw=0.9, label='|YY|', color='tomato', ls='--')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('XX and YY amplitude spectrum (centre pixel)')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# Phase
axes[0, 1].plot(freqs_mhz, np.degrees(np.angle(xx_spec)), lw=0.9, label='∠XX', color='steelblue')
axes[0, 1].plot(freqs_mhz, np.degrees(np.angle(yy_spec)), lw=0.9, label='∠YY', color='tomato', ls='--')
axes[0, 1].set_ylabel('Phase (deg)')
axes[0, 1].set_title('XX and YY phase spectrum (centre pixel)')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# Frequency-averaged amplitude maps
for ax_i, (data, label) in enumerate([(XX, 'XX'), (YY, 'YY')]):
    amp_map = np.abs(data).mean(axis=0)
    im = axes[1, ax_i].imshow(
        amp_map, origin='lower', cmap='inferno',
        extent=extent_deg
    )
    plt.colorbar(im, ax=axes[1, ax_i])
    axes[1, ax_i].set_title(f'|{label}| amplitude map (freq-avg)')
    axes[1, ax_i].set_xlabel('Δx (deg)')
    axes[1, ax_i].set_ylabel('Δy (deg)')

fig.suptitle(
    f'Raw complex data — Antenna {antennas[ANT_IDX]}, Beam {beams[BEAM_IDX]}',
    y=1.01
)
fig.tight_layout()
plt.show()

ratio = np.abs(xx_spec).mean() / np.abs(yy_spec).mean()
print(f'Mean |XX|/|YY| ratio at centre pixel : {ratio:.4f}  (1.0 = perfect balance)')